# AgroVerify Edge — Commodity Image Classifier
**MobileNetV3-Small → ONNX → TFLite INT8**

This notebook trains a 10-class commodity image classifier for the AgroVerify Edge app and exports it as a TFLite INT8 model ready for on-device inference.

**Runtime:** GPU (T4 recommended) — select *Runtime → Change runtime type → T4 GPU*

**Estimated time:**
- Dataset download: ~60–90 min
- Training (30 epochs): ~30–45 min on T4
- Export: ~5 min

**Classes:** maize, cassava, sorghum, rice, soy, groundnuts, yam, millet, cocoa, palm_oil

## 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2 — Mount Google Drive (saves dataset + checkpoints across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/agroverify_classifier'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())

## 3 — Clone repo and install dependencies

In [ ]:
import os

REPO_DIR = os.path.join(WORK_DIR, 'repo')

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/basseyekpenyong/agroverify_edge_project.git "{REPO_DIR}"
else:
    !git -C "{REPO_DIR}" pull
    print('Repo already cloned — pulled latest')

CLASSIFIER_DIR = os.path.join(REPO_DIR, 'ai-models', 'vision', 'commodity_classifier')
os.chdir(CLASSIFIER_DIR)
print('Script directory:', os.getcwd())

In [ ]:
# Install requirements (skip large TF packages already in Colab)
!pip install -q \
    onnx>=1.16.0 \
    onnxruntime>=1.18.0 \
    onnx2tf>=1.20.0 \
    icrawler>=0.6.6 \
    tqdm \
    Pillow

print('✅ Dependencies installed')

## 4 — Download Dataset

Downloads ~600 images per class (6,000 total) from iNaturalist API + Bing supplement.  
Stored in Google Drive so it survives session restarts.

In [ ]:
import os

# Symlink dataset dir to Drive so downloads persist
DATASET_DRIVE = os.path.join(WORK_DIR, 'datasets')
DATASET_LOCAL = os.path.join(CLASSIFIER_DIR, 'datasets')
os.makedirs(DATASET_DRIVE, exist_ok=True)

if not os.path.exists(DATASET_LOCAL):
    os.symlink(DATASET_DRIVE, DATASET_LOCAL)
    print('Dataset symlinked to Drive')
else:
    print('Dataset directory already exists')

# Check how many images already downloaded
from pathlib import Path
raw_dir = Path(DATASET_LOCAL) / 'commodity_images' / 'raw'
if raw_dir.exists():
    for cls in sorted(raw_dir.iterdir()):
        n = len(list(cls.glob('*.jpg')))
        print(f'  {cls.name:12s}: {n} images')
else:
    print('No images yet — running download...')

In [ ]:
# Download dataset — ~60-90 min for 6,000 images
# Re-run safely: skips classes already at target count
!python download_dataset.py --per-class 600

In [ ]:
# Verify split
from pathlib import Path
dataset_dir = Path('datasets/commodity_images')
for split in ['train', 'val', 'test']:
    total = sum(len(list((dataset_dir / split / cls).glob('*.jpg')))
                for cls in (dataset_dir / split).iterdir()
                if (dataset_dir / split / cls).is_dir())
    print(f'{split:6s}: {total} images')

## 5 — Train MobileNetV3-Small

~30–45 minutes on T4 GPU. Best checkpoint auto-saved when validation accuracy improves.

In [ ]:
import os

# Symlink checkpoints to Drive for persistence
CKPT_DRIVE = os.path.join(WORK_DIR, 'checkpoints')
CKPT_LOCAL = os.path.join(CLASSIFIER_DIR, 'checkpoints')
os.makedirs(CKPT_DRIVE, exist_ok=True)

if not os.path.exists(CKPT_LOCAL):
    os.symlink(CKPT_DRIVE, CKPT_LOCAL)
    print('Checkpoints symlinked to Drive')

from pathlib import Path
ckpt = Path(CKPT_LOCAL) / 'commodity_classifier.pt'
if ckpt.exists():
    print(f'Checkpoint already exists ({ckpt.stat().st_size / 1e6:.1f} MB) — training will overwrite if new best is found')

In [ ]:
!python train.py

## 6 — Evaluate on Test Set

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from pathlib import Path

CLASSES = [
    'maize', 'cassava', 'sorghum', 'rice', 'soy',
    'groundnuts', 'yam', 'millet', 'cocoa', 'palm_oil',
]
NUM_CLASSES = len(CLASSES)
IMG_SIZE = 224
CHECKPOINT = Path('checkpoints/commodity_classifier.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device, weights_only=True))
model = model.to(device).eval()

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds = datasets.ImageFolder('datasets/commodity_images/test', transform=test_transform)
test_loader = DataLoader(test_ds, batch_size=32, num_workers=2)

correct = total = 0
class_correct = [0] * NUM_CLASSES
class_total = [0] * NUM_CLASSES

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        for p, l in zip(preds, labels):
            class_correct[l] += (p == l).item()
            class_total[l] += 1

print(f'\nOverall test accuracy: {correct/total*100:.2f}%  ({correct}/{total})')
print(f'\nPer-class accuracy:')
idx_to_class = {v: k for k, v in test_ds.class_to_idx.items()}
for i in range(NUM_CLASSES):
    if class_total[i] > 0:
        print(f'  {idx_to_class[i]:12s}: {class_correct[i]/class_total[i]*100:.1f}%  ({class_correct[i]}/{class_total[i]})')

## 7 — Export to TFLite INT8

Converts the best checkpoint to a TFLite INT8 model using `onnx2tf`.  
Uses real validation images for accurate INT8 calibration.

In [ ]:
# Symlink exports to Drive
import os
EXPORTS_DRIVE = os.path.join(WORK_DIR, 'exports')
EXPORTS_LOCAL = os.path.join(CLASSIFIER_DIR, 'exports')
os.makedirs(EXPORTS_DRIVE, exist_ok=True)
if not os.path.exists(EXPORTS_LOCAL):
    os.symlink(EXPORTS_DRIVE, EXPORTS_LOCAL)

!python export_tflite.py

## 8 — Verify TFLite Model

In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path

TFLITE_PATH = 'exports/commodity_classifier_int8.tflite'

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

print('Model info:')
print(f'  File size : {Path(TFLITE_PATH).stat().st_size / 1024:.0f} KB')
print(f'  Input     : {inp["shape"]}  dtype={inp["dtype"].__name__}')
print(f'  Output    : {out["shape"]}  dtype={out["dtype"].__name__}')

# Quick sanity inference
dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
interpreter.set_tensor(inp['index'], dummy)
interpreter.invoke()
scores = interpreter.get_tensor(out['index'])[0]

CLASSES = ['maize','cassava','sorghum','rice','soy','groundnuts','yam','millet','cocoa','palm_oil']
top = np.argmax(scores)
print(f'\nSanity check (random input): predicted "{CLASSES[top]}" — model responds ✅')

## 9 — Download TFLite file to your computer

This downloads the model to your local machine.  
Place it in `mobile-app/assets/models/commodity_classifier_int8.tflite`.

In [ ]:
from google.colab import files
files.download('exports/commodity_classifier_int8.tflite')
files.download('exports/labels.txt')
print('Downloaded: commodity_classifier_int8.tflite + labels.txt')
print()
print('Place the .tflite file in:')
print('  mobile-app/assets/models/commodity_classifier_int8.tflite')
print()
print('Then run: flutter run')

---
## Summary

| Step | Output |
|---|---|
| Dataset | ~6,000 images, 10 classes, 70/20/10 split |
| Model | MobileNetV3-Small, 30 epochs |
| Target accuracy | ≥ 85% top-1 on test set |
| TFLite size | < 5 MB INT8 |
| Inference target | < 2s on mid-range Android |

All outputs saved to Google Drive under `agroverify_classifier/` and persist across Colab sessions.